In [1]:
import cv2
import numpy as np
from PIL import Image
import torch
from torchvision import transforms

import os
import json
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import datasets

import torch
import torch.nn as nn
import torch.optim as optim
from collections import Counter
from torchvision import models
from tqdm import tqdm
from pathlib import Path

In [2]:
class ColorConstancy(object):
    """
    Implements a simple 'Gray World' Color Constancy algorithm.
    It assumes the average color of the scene should be neutral gray.
    """
    def __call__(self, img):
        # Convert PIL to CV2 (OpenCV uses BGR)
        img_np = np.array(img)
        img_bgr = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)

        # Calculate channel averages
        b_avg = np.mean(img_bgr[:, :, 0])
        g_avg = np.mean(img_bgr[:, :, 1])
        r_avg = np.mean(img_bgr[:, :, 2])

        # Gray world assumption
        avg_gray = (b_avg + g_avg + r_avg) / 3

        # Scaling factors
        b_scale = avg_gray / (b_avg + 1e-6)
        g_scale = avg_gray / (g_avg + 1e-6)
        r_scale = avg_gray / (r_avg + 1e-6)

        # Apply scaling
        img_bgr[:, :, 0] = np.clip(img_bgr[:, :, 0] * b_scale, 0, 255)
        img_bgr[:, :, 1] = np.clip(img_bgr[:, :, 1] * g_scale, 0, 255)
        img_bgr[:, :, 2] = np.clip(img_bgr[:, :, 2] * r_scale, 0, 255)

        # Convert back to RGB and PIL
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        return Image.fromarray(img_rgb.astype('uint8'))

# Define the Robust Augmentation Pipeline
def get_robust_transforms(img_size=224, normalize=True):
    stats = {'mean':[0.485,0.456,0.406], 'std':[0.229,0.224,0.225]}

    train_t = [
        transforms.Resize((256, 256)),
        transforms.RandomCrop((img_size, img_size)),

        ColorConstancy(),                 # 1. Color Constancy
        transforms.RandAugment(num_ops=2, magnitude=9), # 2. RandAugment

        transforms.ToTensor()
    ]

    val_t = [
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor()
    ]

    if normalize:
        train_t.append(transforms.Normalize(**stats))
        val_t.append(transforms.Normalize(**stats))

    return transforms.Compose(train_t), transforms.Compose(val_t)

In [3]:
class UnifiedMixedDataset(Dataset):
    def __init__(self, pv_dir, pd_dir, classmap_path, transform=None):
        self.transform = transform
        self.samples = [] # List of (path, unified_label_index)
        self.unified_names = []

        # 1. Load Classmap
        with open(classmap_path, 'r') as f:
            cm = json.load(f)
        pv_map = cm["plantvillage_to_unified"]
        pd_map = cm["plantdoc_to_unified"]

        # 2. Find Intersecting Classes (Classes that exist in BOTH)
        # We only want to train on classes where we have both Lab and Field data
        pv_classes = set(pv_map.values())
        pd_classes = set(pd_map.values())
        common_classes = sorted(list(pv_classes.intersection(pd_classes)))

        # Create a map: unified_name -> integer_index
        self.class_to_idx = {name: i for i, name in enumerate(common_classes)}
        self.unified_names = common_classes

        print(f"Found {len(common_classes)} overlapping classes between PV and PD.")

        # 3. Collect Images from PlantVillage
        self._collect_images(pv_dir, pv_map, "PlantVillage")

        # 4. Collect Images from PlantDoc
        self._collect_images(pd_dir, pd_map, "PlantDoc")

    def _collect_images(self, root_dir, mapping, source_name):
        count = 0
        if not os.path.exists(root_dir):
            print(f"⚠️ Warning: {source_name} dir not found: {root_dir}")
            return

        for folder in os.listdir(root_dir):
            folder_path = os.path.join(root_dir, folder)
            if not os.path.isdir(folder_path): continue

            # Check if this folder maps to a common unified class
            unified_label = mapping.get(folder)
            if unified_label in self.class_to_idx:
                label_idx = self.class_to_idx[unified_label]

                # Add all images
                for img_file in os.listdir(folder_path):
                    if img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
                        self.samples.append((os.path.join(folder_path, img_file), label_idx))
                        count += 1
        print(f"Loaded {count} images from {source_name}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label

def make_balanced_loader(dataset, batch_size, num_workers=2):
    # Calculate weights for Balanced Sampling
    targets = [s[1] for s in dataset.samples]
    class_counts = Counter(targets)

    # Weight = 1.0 / count (Rare classes get higher weight)
    weights = []
    for t in targets:
        weights.append(1.0 / class_counts[t])

    weights = torch.DoubleTensor(weights)

    # Create the sampler
    sampler = WeightedRandomSampler(weights, len(weights))

    return DataLoader(dataset, batch_size=batch_size, sampler=sampler, num_workers=num_workers, pin_memory=False)

In [4]:
import os
import zipfile

DRIVE_FOLDER = '/content/drive/MyDrive/Colab Notebooks/mobile-leafdoc'
ZIP_PATH_pd = os.path.join(DRIVE_FOLDER, 'PlantDoc-Dataset.zip')
ZIP_PATH_pv = os.path.join(DRIVE_FOLDER, 'PlantVillage-Dataset.zip')

if not os.path.exists('/content/data'):
    print("Unzipping dataset...")
    with zipfile.ZipFile(ZIP_PATH_pd, 'r') as zip_ref:
        zip_ref.extractall('/content/data')
    with zipfile.ZipFile(ZIP_PATH_pv, 'r') as zip_ref:
        zip_ref.extractall('/content/data')
    print("✅ Unzip complete!")
else:
    print("Data already unzipped.")

if os.path.exists('/content/data/PlantVillage-Dataset/raw/color'):
    print("✅ PV - Found Training Folder!")
    PV_DIR = '/content/data/PlantVillage-Dataset/raw/color'
else:
    print("⚠️ Check your zip structure. Could not find 'PlantVillage-Dataset/raw/color'")

if os.path.exists('/content/data/PlantDoc-Dataset/train'):
    print("✅ PD - Found Training Folder!")
    PD_DIR = '/content/data/PlantDoc-Dataset/train'
else:
    print("⚠️ Check your zip structure. Could not find 'PlantDoc-Dataset/train'")

Unzipping dataset...
✅ Unzip complete!
✅ PV - Found Training Folder!
✅ PD - Found Training Folder!


In [5]:
# --- CONFIGURATION ---
# Update these paths to match your Colab/Local setup
# PV_DIR = "PlantVillage-Dataset/raw/color/"
# PD_DIR = "PlantDoc-Dataset/train/"
CLASSMAP_PATH = os.path.join(DRIVE_FOLDER, 'classmap.json')
OUT_DIR = "/content/drive/MyDrive/Colab Notebooks/mobile-leafdoc/runs/mixed_pv_pd/run2"
# OUT_DIR = "runs/mixed_robustness/run2"

BATCH_SIZE = 64
EPOCHS = 15
LR = 1e-4
IMG_SIZE = 224
SEED = 42

In [6]:
train_tfm, val_tfm = get_robust_transforms(IMG_SIZE)

In [7]:
print("Building Mixed Dataset...")
# combined dataset for training
ds_train = UnifiedMixedDataset(PV_DIR, PD_DIR, CLASSMAP_PATH, transform=train_tfm)

Building Mixed Dataset...
Found 28 overlapping classes between PV and PD.
Loaded 38542 images from PlantVillage
Loaded 2336 images from PlantDoc


In [8]:
# load PlantDoc separately as the validation set
ds_val = datasets.ImageFolder(PD_DIR, transform=val_tfm)

# balance train data
train_loader = make_balanced_loader(ds_train, BATCH_SIZE)

In [9]:
# Setup Model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
num_classes = len(ds_train.class_to_idx)
print(f"Training on {num_classes} unified classes.")

Training on 28 unified classes.


In [10]:
# Load ImageNet Weights
model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
model.classifier[3] = nn.Linear(model.classifier[3].in_features, num_classes)
model = model.to(device)

Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:00<00:00, 95.6MB/s]


In [11]:
# Training Loop
opt = optim.Adam(model.parameters(), lr=LR)
ce = nn.CrossEntropyLoss()

Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
best_loss = float('inf')

In [15]:
for epoch in range(1, EPOCHS+1):
    model.train()
    loss_accum = 0.0
    correct = 0
    total = 0

    for x, y in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}", leave=False):
        x, y = x.to(device), y.to(device)

        opt.zero_grad()
        logits = model(x)
        loss = ce(logits, y)
        loss.backward()
        opt.step()

        loss_accum += loss.item()
        correct += (logits.argmax(1) == y).sum().item()
        total += y.size(0)

    train_acc = correct / total
    avg_loss = loss_accum / len(train_loader)

    print(f"Epoch {epoch}: Train Loss={avg_loss:.4f} | Train Acc={train_acc:.4f}")

    # Save checkpoint
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save({
            'model': model.state_dict(),
            'classes': ds_train.unified_names, # Save the unified names!
            'class_to_idx': ds_train.class_to_idx
        }, f"{OUT_DIR}/mixed_model_best.pt")
    torch.save({
            'model': model.state_dict(),
            'classes': ds_train.unified_names,
            'class_to_idx': ds_train.class_to_idx
        }, f"{OUT_DIR}/mixed_model_final.pt")

Epoch 1: Train Loss=0.0521 | Train Acc=0.9827


Epoch 2: Train Loss=0.0472 | Train Acc=0.9850


Epoch 3: Train Loss=0.0406 | Train Acc=0.9870


Epoch 4: Train Loss=0.0388 | Train Acc=0.9876


Epoch 5: Train Loss=0.0383 | Train Acc=0.9875


...

Epoch 11: Train Loss=0.0257 | Train Acc=0.9916


Epoch 12: Train Loss=0.0243 | Train Acc=0.9920


Epoch 13: Train Loss=0.0220 | Train Acc=0.9928


Epoch 14: Train Loss=0.0224 | Train Acc=0.9925


Epoch 15: Train Loss=0.0202 | Train Acc=0.9934
